__Проект по исследованию читательского ресурса.__  

Представлены данные о книгах, авторах, издательствах, читательских рейтингах и рецензиях, размещенных на изучаемом ресурсе.  

Цели  
    
    Оценить книжный рынок.
    Активность и поведение пользователей.
    Выявить лидеров среди издательств.
    Популярные жанры и авторы.
    Определить уровень рейтинга и количество обзоров.

In [ ]:
# импортируем библиотеки
import pandas as pd
import sqlalchemy as sa

In [ ]:
# устанавливаем параметры
db_config = {
'user': 'praktikum_student', # имя пользователя
'pwd': 'Sdf4$2;d-d30pp', # пароль
'host': 'rc1b-wcoijxj3yxfsf3fs.mdb.yandexcloud.net',
'port': 6432, # порт подключения
'db': 'data-analyst-final-project-db' # название базы данных
}
connection_string = 'postgresql://{user}:{pwd}@{host}:{port}/{db}'.format(**db_config)

In [ ]:
# сохраняем коннектор
engine = sa.create_engine(connection_string, connect_args={'sslmode':'require'})
# чтобы выполнить SQL-запрос, пишем функцию с использованием Pandas

In [ ]:
def get_sql_data(query:str, engine:sa.engine.base.Engine=engine) -> pd.DataFrame:
    '''Открываем соединение, получаем данные из sql, закрываем соединение'''
    with engine.connect() as con:
        return pd.read_sql(sql=sa.text(query), con = con)

In [ ]:
# формируем запрос и выводим данные
query = '''SELECT * FROM books LIMIT 5'''
get_sql_data(query)

,book_id,author_id,title,num_pages,publication_date,publisher_id
0,1,546,'Salem's Lot,594,2005-11-01,93
1,2,465,1 000 Places to See Before You Die,992,2003-05-22,336
2,3,407,13 Little Blue Envelopes (Little Blue Envelope...,322,2010-12-21,135
3,4,82,1491: New Revelations of the Americas Before C...,541,2006-10-10,309
4,5,125,1776,386,2006-07-04,268


In [ ]:
# функция для просмотра всех таблиц, их размеров и вывода первых строк
def f(table):
    print(f'---------описание таблицы "{table}"---------')

    query = '''SELECT COUNT(*) FROM table'''.replace('table', table)
    res = get_sql_data(query)
    n_str = res['count'].iloc[0]
    print(f'количество строк: {n_str}')

    # можно было сразу выгрузить все данные и их смотреть, но в задании написано, что с помощью запросов
    query = '''SELECT * FROM table LIMIT 5'''.replace('table', table)
    res = get_sql_data(query)
    print(f'колонки: {res.columns}')

    display(res)
    print('')
#     print(f'------------------')

In [ ]:
for table in ['books', 'authors', 'publishers', 'ratings', 'reviews']:
    f(table)

---------описание таблицы "books"---------
количество строк: 1000
колонки: Index(['book_id', 'author_id', 'title', 'num_pages', 'publication_date',
       'publisher_id'],
      dtype='object')


,book_id,author_id,title,num_pages,publication_date,publisher_id
0,1,546,'Salem's Lot,594,2005-11-01,93
1,2,465,1 000 Places to See Before You Die,992,2003-05-22,336
2,3,407,13 Little Blue Envelopes (Little Blue Envelope...,322,2010-12-21,135
3,4,82,1491: New Revelations of the Americas Before C...,541,2006-10-10,309
4,5,125,1776,386,2006-07-04,268



---------описание таблицы "authors"---------
количество строк: 636
колонки: Index(['author_id', 'author'], dtype='object')


,author_id,author
0,1,A.S. Byatt
1,2,Aesop/Laura Harris/Laura Gibbs
2,3,Agatha Christie
3,4,Alan Brennert
4,5,Alan Moore/David Lloyd



---------описание таблицы "publishers"---------
количество строк: 340
колонки: Index(['publisher_id', 'publisher'], dtype='object')


,publisher_id,publisher
0,1,Ace
1,2,Ace Book
2,3,Ace Books
3,4,Ace Hardcover
4,5,Addison Wesley Publishing Company



---------описание таблицы "ratings"---------
количество строк: 6456
колонки: Index(['rating_id', 'book_id', 'username', 'rating'], dtype='object')


,rating_id,book_id,username,rating
0,1,1,ryanfranco,4
1,2,1,grantpatricia,2
2,3,1,brandtandrea,5
3,4,2,lorichen,3
4,5,2,mariokeller,2



---------описание таблицы "reviews"---------
количество строк: 2793
колонки: Index(['review_id', 'book_id', 'username', 'text'], dtype='object')


,review_id,book_id,username,text
0,1,1,brandtandrea,Mention society tell send professor analysis. ...
1,2,1,ryanfranco,Foot glass pretty audience hit themselves. Amo...
2,3,2,lorichen,Listen treat keep worry. Miss husband tax but ...
3,4,3,johnsonamanda,Finally month interesting blue could nature cu...
4,5,3,scotttamara,Nation purpose heavy give wait song will. List...


Данные представленные пятью таблицами:  


   "books"      количество строк: 1000 колонки 'book_id', 'author_id', 'title', 'num_pages', 'publication_date', 'publisher_id'

   "authors"    количество строк: 363  колонки 'author_id', 'author'

   "publishers" количество строк: 340  колонки 'publisher_id', 'publisher'

   "ratings"    количество строк: 6456 колонки 'rating_id', 'book_id', 'username', 'rating'

   "reviews"    количество строк: 2793 колонки 'review_id', 'book_id', 'username', 'text'


Посчитать, сколько книг вышло после 1 января 2000 года

In [ ]:
#
query = '''SELECT COUNT(*) AS num_books_after_2000
FROM books
WHERE publication_date > ('2000-01-01') '''
get_sql_data(query)

,num_books_after_2000
0,819


После 2000 вышло 819 книег.

Для каждой книги посчитать количество обзоров и среднюю оценку

In [ ]:
#

get_sql_data(
'''SELECT
    b.book_id,
    b.title,
    COUNT(DISTINCT r.review_id) AS review_count,
    AVG(rat.rating) AS average_rating,
    MAX(AVG(rat.rating)) OVER () AS average_rating_max,
    MIN(AVG(rat.rating)) OVER () AS average_rating_min
FROM
    books b
LEFT JOIN
    reviews r ON b.book_id = r.book_id
LEFT JOIN
    ratings rat ON b.book_id = rat.book_id
GROUP BY
    b.book_id, b.title
ORDER BY average_rating DESC, review_count  DESC;
'''
)

,book_id,title,review_count,average_rating,average_rating_max,average_rating_min
0,17,A Dirty Job (Grim Reaper #1),4,5.00,5.0,1.5
1,553,School's Out—Forever (Maximum Ride #2),3,5.00,5.0,1.5
2,444,Moneyball: The Art of Winning an Unfair Game,3,5.00,5.0,1.5
3,86,Arrows of the Queen (Heralds of Valdemar #1),2,5.00,5.0,1.5
4,972,Wherever You Go There You Are: Mindfulness Me...,2,5.00,5.0,1.5
...,...,...,...,...,...,...
995,915,The World Is Flat: A Brief History of the Twen...,3,2.25,5.0,1.5
996,202,Drowning Ruth,3,2.00,5.0,1.5
997,316,His Excellency: George Washington,2,2.00,5.0,1.5
998,371,Junky,2,2.00,5.0,1.5


Средняя оценка книг варьируется от 1.5 до 5 баллов.

<div class="alert alert-info" style="border-radius: 15px; box-shadow: 4px 4px 4px; border: 1px solid " ><b> Комментарий студента : </b>

В запросе сортировка по этому значению, выедены первая и последняя строчка. Проверю еще оконной функцией.  
    
Думал не сработает, если засунуть в оконную функцию агрегированное поле, а нет работает. :)
    
В 1С тоже есть язык запросов, только на русском и нет оконных функций. А это очень мощный инструмент и может сэкономить много сил и времени.   

</div>

Определить издательство, которое выпустило наибольшее число книг толще 50 страниц — так вы исключите из анализа брошюры

In [ ]:
#
get_sql_data(
'''
WITH publisher_num_books AS(
SELECT
    p.publisher,
    COUNT(b.book_id) AS num_books

FROM
    books b
JOIN
    publishers p ON b.publisher_id = p.publisher_id
WHERE
    b.num_pages > 50
GROUP BY
    p.publisher
ORDER BY
    num_books DESC)


SELECT *,
    ROUND(100*num_books/SUM(num_books) OVER (), 2) AS num_books_dolya
FROM
    publisher_num_books
LIMIT 1;
'''
)

,publisher,num_books,num_books_dolya
0,Penguin Books,42,4.23


Больше всего, 42 книги, издано издательством Penguin Books.

Определить автора с самой высокой средней оценкой книг — учитывайте  только книги с 50 и более оценками

In [ ]:
#
get_sql_data(
'''
WITH book_ratings AS (
    SELECT
        b.book_id,
        b.author_id,
        AVG(r.rating) AS avg_rating,
        COUNT(r.rating_id) AS num_ratings
    FROM
        books b
    JOIN
        ratings r ON b.book_id = r.book_id
    GROUP BY
        b.book_id
    HAVING
        COUNT(r.rating_id) >= 50
),
author_ratings AS (
    SELECT
        a.author_id,
        a.author,
        AVG(br.avg_rating) AS avg_author_rating
    FROM
        book_ratings br
    JOIN
        authors a ON br.author_id = a.author_id
    GROUP BY
        a.author_id, a.author
)
SELECT
    author_id,
    author,
    avg_author_rating
FROM
    author_ratings
ORDER BY
    avg_author_rating DESC
LIMIT 1;
'''
)

,author_id,author,avg_author_rating
0,236,J.K. Rowling/Mary GrandPré,4.283844


Популярная писательница с самой высокой средней оценкой - J.K. Rowling (Mary GrandPré).

Посчитать среднее количество обзоров от пользователей, которые поставили больше 48 оценок

In [ ]:
# В подзапросе выберем пользователей, которые поставили больше 48 оценок
# Во временной таблице по этим пользователям посчитаем количество обзоров
# В основном запросе получим среднее значение для количества обзоров

get_sql_data(
'''
WITH reviews_count_by_user AS(
SELECT
    username,
    COUNT(r.review_id) AS num_reviews

FROM
    reviews r
WHERE
    r.username IN (
                SELECT
                    DISTINCT (r.username)
                FROM
                    ratings r
                GROUP BY
                    r.username
                HAVING
                    COUNT(r.rating_id)>48
                    )
GROUP BY
    r.username
)

SELECT
    AVG(num_reviews) AS avg_reviews
FROM
    reviews_count_by_user;

'''
)

,avg_reviews
0,24.0


Активные пользователи пишут в среднем 24 обзора.

Результаты исследования.

В новом столетии опубликовали 819 книг.  
Книги с высоким рейтингом не всегда популярны среди критиков.  
Самое большое количество 42 книги опубликовано издательством «Penguin Books».  
Популярный автор с самым высоки рейтингом 4.3 балла Джоа́н Ро́улинг  британская писательница,  автор серии романов о Гарри Поттере.   
Пользователи, поставившие более 48 оценок, написали в среднем 24 рецензии.

Выводы.  

В последнее время публикуется достаточно много книг. Книгам дают оценки как профессиональные критики, так и простые читатели выставляют рейтинги и пишут обзоры. Мнение профессионалов не всегда совпадает с народным мнением.  

Самое крупное издательство Penguin Books. На него приходится больше 4% всех изданных книг.
Высокие рейтинги имеют авторы подросткового фентези. Пользователи активны не только в выставлении оценок, но и любят поделиться своим мнением о книге.  

Сегодня книга — это не только писатель, издательство и читатели, но и целые сообщества критиков, фанатов, обзорщиков, общающихся между собой на различных платформах.            


На основе полученных данных можно рекомендовать:
    
    разработать систему рейтингов для активных пользователей
    систему коммуникации читателей и критиков
    возможно размещение своих произведений начинающими писателями на суд читателей и критиков
    плотно работать с крупными издателями и популярными писателями
